# 🏨 Phase 5 — KPI Dashboard
**Goal:** Build an interactive Quality dashboard with Plotly — mimics a real hotel Quality Monitoring system  
**KPIs:** Average Score · NPS · Complaint Rate · Sentiment Score · Segment Performance

---

## 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re, warnings
warnings.filterwarnings('ignore')

print('✅ Libraries loaded')

✅ Libraries loaded


## 2. Load & Prepare All Data

In [2]:
df = pd.read_csv(r"C:\Users\dimma\OneDrive\Υπολογιστής\Hotel 515\Hotel_Reviews.csv")
df['Negative_Review'] = df['Negative_Review'].replace('No Negative', '')
df['Positive_Review'] = df['Positive_Review'].replace('No Positive', '')
df['Hotel_Country'] = df['Hotel_Address'].apply(lambda x: x.split()[-1])
df['Review_Date'] = pd.to_datetime(df['Review_Date'])
df['YearMonth'] = df['Review_Date'].dt.to_period('M').astype(str)
df['Review_Month'] = df['Review_Date'].dt.month
df['Review_Year'] = df['Review_Date'].dt.year

# NPS simulation: Score >= 9 = Promoter, 7-8 = Passive, <7 = Detractor
def nps_category(score):
    if score >= 9:   return 'Promoter'
    elif score >= 7: return 'Passive'
    else:            return 'Detractor'

df['nps_category'] = df['Reviewer_Score'].apply(nps_category)

def calc_nps(series):
    cats = series.value_counts(normalize=True) * 100
    promoters  = cats.get('Promoter', 0)
    detractors = cats.get('Detractor', 0)
    return round(promoters - detractors, 1)

overall_nps = calc_nps(df['nps_category'])

# Complaint keywords categories (reuse from Phase 2)
CATEGORY_KEYWORDS = {
    'Cleanliness': ['dirty','clean','stain','dust','smell','odor','mold','filthy','bathroom','bugs'],
    'Service':     ['staff','rude','unfriendly','slow','service','reception','attitude','wait'],
    'Food & Bev':  ['breakfast','food','restaurant','bar','dinner','meal','buffet','cold food'],
    'Noise':       ['noise','noisy','loud','party','music','street','sleep','disturb'],
    'Room':        ['room','bed','mattress','small','dark','cramped','outdated','window'],
    'Location':    ['location','far','transport','parking','access','area','remote'],
    'Value':       ['expensive','overpriced','price','value','rip off','not worth','fee'],
    'Facilities':  ['pool','gym','wifi','elevator','spa','air conditioning','heating','broken']
}

def categorize(text):
    if not isinstance(text, str): return 'Other'
    text_l = text.lower()
    for cat, kws in CATEGORY_KEYWORDS.items():
        if any(k in text_l for k in kws):
            return cat
    return 'Other'

df['complaint_cat'] = df['Negative_Review'].apply(categorize)

print(f'✅ Data ready. Shape: {df.shape}')
print(f'📊 Overall NPS: {overall_nps}')

✅ Data ready. Shape: (515738, 23)
📊 Overall NPS: 31.1


## 3. KPI Summary Cards

In [4]:
avg_score      = df['Reviewer_Score'].mean()
pct_promoters  = (df['nps_category'] == 'Promoter').mean() * 100
pct_detractors = (df['nps_category'] == 'Detractor').mean() * 100
complaint_rate = (df['Negative_Review'].str.len() > 10).mean() * 100

fig = go.Figure()

kpis = [
    ('⭐ Avg Score',       f'{avg_score:.2f}/10',      'rgb(26,118,255)'),
    ('📈 NPS Score',       f'{overall_nps}',            'rgb(46,184,46)' if overall_nps > 0 else 'rgb(220,50,50)'),
    ('😊 Promoters',       f'{pct_promoters:.1f}%',     'rgb(46,184,46)'),
    ('😞 Detractors',      f'{pct_detractors:.1f}%',    'rgb(220,50,50)'),
    ('⚠️ Complaint Rate',  f'{complaint_rate:.1f}%',    'rgb(255,165,0)'),
    ('📝 Total Reviews',   f'{len(df):,}',              'rgb(150,100,200)')
]

for i, (label, value, color) in enumerate(kpis):
    col = i % 3
    row = i // 3
    fig.add_annotation(
        x=col/3 + 0.17, y=1 - row*0.55,
        text=f'<b style="font-size:22px;color:{color}">{value}</b><br><span style="font-size:13px;color:gray">{label}</span>',
        showarrow=False,
        align='center',
        bgcolor='rgba(240,240,240,0.9)',
        bordercolor=color,
        borderwidth=2,
        borderpad=12,
        xref='paper', yref='paper'
    )

fig.update_layout(
    title='🏨 Hotel Quality — KPI Summary',
    height=300,
    xaxis=dict(visible=False), yaxis=dict(visible=False),
    plot_bgcolor='grey', paper_bgcolor='white'
)
fig.show()

## 4. Score Trend Over Time (Interactive)

In [5]:
monthly = df.groupby('YearMonth').agg(
    avg_score=('Reviewer_Score', 'mean'),
    review_count=('Reviewer_Score', 'count'),
    nps=('nps_category', lambda x: calc_nps(x))
).reset_index()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Average Score Over Time', 'Monthly NPS Score'),
                    row_heights=[0.6, 0.4])

fig.add_trace(go.Scatter(
    x=monthly['YearMonth'], y=monthly['avg_score'],
    mode='lines+markers', name='Avg Score',
    line=dict(color='royalblue', width=2),
    hovertemplate='%{x}<br>Score: %{y:.2f}<extra></extra>'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=monthly['YearMonth'], y=monthly['nps'],
    name='NPS',
    marker_color=['seagreen' if v >= 0 else 'tomato' for v in monthly['nps']],
    hovertemplate='%{x}<br>NPS: %{y:.1f}<extra></extra>'
), row=2, col=1)

fig.update_layout(
    title='📈 Score & NPS Trend Over Time',
    height=550, showlegend=True,
    plot_bgcolor='rgba(240,240,240,0.5)'
)
fig.update_yaxes(title_text='Avg Score', range=[7, 9], row=1, col=1)
fig.update_yaxes(title_text='NPS', row=2, col=1)
fig.show()

## 5. NPS Gauge Chart

In [6]:
fig = go.Figure(go.Indicator(
    mode='gauge+number+delta',
    value=overall_nps,
    title={'text': 'Net Promoter Score (NPS)', 'font': {'size': 20}},
    delta={'reference': 0, 'increasing': {'color': 'green'}, 'decreasing': {'color': 'red'}},
    gauge={
        'axis': {'range': [-100, 100], 'tickwidth': 1},
        'bar': {'color': 'royalblue'},
        'steps': [
            {'range': [-100, 0],  'color': 'rgba(255,80,80,0.3)'},
            {'range': [0, 50],    'color': 'rgba(255,200,0,0.3)'},
            {'range': [50, 100],  'color': 'rgba(0,200,0,0.3)'}
        ],
        'threshold': {
            'line': {'color': 'red', 'width': 3},
            'thickness': 0.75,
            'value': 0
        }
    }
))
fig.update_layout(height=350)
fig.show()

## 6. Complaint Category Breakdown (Interactive)

In [7]:
complaint_complaints = df[df['Negative_Review'].str.len() > 10]
cat_counts = complaint_complaints['complaint_cat'].value_counts().reset_index()
cat_counts.columns = ['Category', 'Count']

cat_score = complaint_complaints.groupby('complaint_cat')['Reviewer_Score'].mean().reset_index()
cat_score.columns = ['Category', 'Avg_Score']
cat_data = cat_counts.merge(cat_score, on='Category').sort_values('Count', ascending=False)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Complaint Volume per Department', 'Avg Score when Dept Mentioned'),
                    specs=[[{'type': 'bar'}, {'type': 'bar'}]])

fig.add_trace(go.Bar(
    x=cat_data['Category'], y=cat_data['Count'],
    marker_color='tomato', name='Count',
    hovertemplate='%{x}<br>Complaints: %{y:,}<extra></extra>'
), row=1, col=1)

score_sorted = cat_data.sort_values('Avg_Score')
fig.add_trace(go.Bar(
    x=score_sorted['Category'], y=score_sorted['Avg_Score'],
    marker_color=['tomato' if v < 6 else 'orange' if v < 7 else 'seagreen'
                  for v in score_sorted['Avg_Score']],
    name='Avg Score',
    hovertemplate='%{x}<br>Avg Score: %{y:.2f}<extra></extra>'
), row=1, col=2)

fig.update_layout(title='📊 Complaint Department Analysis', height=450, showlegend=False)
fig.update_yaxes(range=[0, 10], row=1, col=2)
fig.show()

## 7. Country Benchmarking Map

In [8]:
country_stats = df.groupby('Hotel_Country').agg(
    avg_score=('Reviewer_Score', 'mean'),
    total_reviews=('Reviewer_Score', 'count'),
    nps=('nps_category', lambda x: calc_nps(x))
).reset_index()

# Map country names to ISO codes for choropleth
country_iso = {
    'Kingdom': 'GBR', 'Netherlands': 'NLD', 'France': 'FRA',
    'Spain': 'ESP', 'Italy': 'ITA', 'Austria': 'AUT'
}
country_stats['iso'] = country_stats['Hotel_Country'].map(country_iso)

fig = px.choropleth(
    country_stats.dropna(subset=['iso']),
    locations='iso',
    color='avg_score',
    hover_name='Hotel_Country',
    hover_data={'avg_score': ':.2f', 'total_reviews': ':,', 'nps': True},
    color_continuous_scale='RdYlGn',
    scope='europe',
    title='🌍 Average Hotel Score by Country'
)
fig.update_layout(height=450)
fig.show()

## 8. Top 15 Hotels Leaderboard

In [9]:
hotel_board = df.groupby('Hotel_Name').agg(
    avg_score=('Reviewer_Score', 'mean'),
    total_reviews=('Reviewer_Score', 'count'),
    country=('Hotel_Country', 'first'),
    nps=('nps_category', lambda x: calc_nps(x))
).reset_index()

top15 = hotel_board[hotel_board['total_reviews'] >= 100].nlargest(15, 'avg_score')
top15['short_name'] = top15['Hotel_Name'].str[:40]

fig = px.bar(
    top15.sort_values('avg_score'),
    x='avg_score', y='short_name',
    color='country', orientation='h',
    text='avg_score',
    hover_data=['total_reviews', 'nps'],
    title='🏆 Top 15 Hotels Leaderboard (min. 100 reviews)',
    labels={'avg_score': 'Average Score', 'short_name': 'Hotel'}
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(height=550, xaxis_range=[8, 10])
fig.show()

## 9. Export HTML Dashboard

In [10]:
# Combine all charts into one standalone HTML file
import plotly.io as pio

# Recreate all figures
figs = []

# 1. Score distribution
figs.append(px.histogram(df, x='Reviewer_Score', nbins=20, title='Score Distribution',
                         color_discrete_sequence=['royalblue']))

# 2. Monthly trend
figs.append(px.line(monthly, x='YearMonth', y='avg_score',
                    title='Score Trend', markers=True))

# 3. Complaints
figs.append(px.bar(cat_data, x='Category', y='Count', color='Avg_Score',
                   color_continuous_scale='RdYlGn',
                   title='Complaints by Department'))

# 4. Country map
fig_map = px.choropleth(
    country_stats.dropna(subset=['iso']),
    locations='iso', color='avg_score',
    hover_name='Hotel_Country',
    color_continuous_scale='RdYlGn',
    scope='europe', title='Avg Score by Country'
)
figs.append(fig_map)

# Write all to single HTML
html_parts = []
html_parts.append('<html><head><title>Hotel Quality Dashboard</title></head><body>')
html_parts.append('<h1 style="font-family:Arial;text-align:center;">🏨 Hotel Quality Intelligence Dashboard</h1>')
for fig in figs:
    html_parts.append(pio.to_html(fig, full_html=False, include_plotlyjs='cdn'))
html_parts.append('</body></html>')

with open('../dashboard/quality_dashboard.html', 'w', encoding='utf-8') as f:
    f.write('\n'.join(html_parts))

print('✅ Dashboard exported: dashboard/quality_dashboard.html')
print()
print('=' * 55)
print('🎉 ALL PHASES COMPLETE!')
print('=' * 55)
print('📁 Project structure:')
print('  notebooks/01_EDA.ipynb')
print('  notebooks/02_Complaint_Categorization.ipynb')
print('  notebooks/03_Sentiment_Analysis.ipynb')
print('  notebooks/04_Guest_Segmentation.ipynb')
print('  notebooks/05_KPI_Dashboard.ipynb  ← YOU ARE HERE')
print('  dashboard/quality_dashboard.html  ← INTERACTIVE DASHBOARD')
print('  data/complaints_categorized.csv')

FileNotFoundError: [Errno 2] No such file or directory: '../dashboard/quality_dashboard.html'